# 推特航空評論三分類情緒分析（2026 版）

## 學習目標

1. 以 HuggingFace Hub 取代本機 / Google Drive 路徑，確保環境可重現
2. 使用 `datasets` 3.x 管線：`load_dataset` → `map(batched=True)` → `DataCollatorWithPadding`（動態 padding）
3. 修正 **資料洩漏** 錯誤：`MinMaxScaler` 必須在 train/test 分割之後才能 fit
4. 以 `train_test_split(stratify_by_column='label', seed=42)` 解決類別不平衡問題
5. 以 `Trainer` + 完整 `TrainingArguments`（bf16、AdamW fused、cosine scheduler）取代手刻訓練迴圈
6. 以 `evaluate.combine` + `compute_metrics` 回呼取代手寫評測迴圈
7. 統一裝置載入：`device_map='auto'` + `torch_dtype=torch.bfloat16`
8. 儲存含 `id2label` / `label2id` 的完整 model card，示範 `push_to_hub`

## 與相鄰 Notebook 的銜接

- 上一個：`../04tokenizer/` — Tokenizer 原理與 `apply_chat_template`
- 下一個：`../06trainer/` — 進階 TrainingArguments 與 SFTTrainer 指令微調

## 前置知識

- HuggingFace `AutoTokenizer` / `AutoModelForSequenceClassification` 基本用法
- PyTorch tensor 基礎
- 分類問題的 Precision / Recall / F1 概念

In [ ]:
# Cell 0 — 版本鎖定（安裝前確認）
# 若在 Colab 或全新虛擬環境執行，取消下一行的 # 號
# !pip install "transformers>=4.46" "datasets>=3.0" "evaluate>=0.4" \
#              "accelerate>=1.0" "safetensors>=0.4" "scikit-learn>=1.5" \
#              "torch>=2.4" -q

## Step 1  匯入函式庫與確認環境

2026 慣例：集中在最上方一次匯入所有需要的模組，方便閱讀與排錯。

In [ ]:
import os
import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.preprocessing import MinMaxScaler

# 固定所有隨機種子，確保結果可重現
set_seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2  載入資料集

使用 `load_dataset` 直接從 HuggingFace Hub 載入，任何環境皆可重現。

```python
dataset = load_dataset('osanseviero/twitter-airline-sentiment')
```

`load_dataset` 會在本機快取（`~/.cache/huggingface/datasets`），第二次呼叫直接讀快取，不需重新下載。

In [ ]:
raw_dataset = load_dataset("osanseviero/twitter-airline-sentiment")
print(raw_dataset)
print("\n欄位：", raw_dataset["train"].column_names)
print("第一筆：", raw_dataset["train"][0])

## Step 3  資料前處理

### 3-1  Label encoding（情緒 → 整數標籤）

原始資料集只有 `train` split，我們稍後手動切割。

In [ ]:
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
LABEL2ID  = LABEL_MAP

def add_label(example):
    example["label"] = LABEL_MAP[example["airline_sentiment"]]
    return example

ds = raw_dataset["train"].map(add_label)
print(ds.features)
print("\n標籤分布：", {k: ds["label"].count(v) for k, v in LABEL_MAP.items()})

### 3-2  挑選特徵欄位並補空缺值

本 notebook 的教學重點是多特徵融合——把數值特徵（信心度、轉推數）序列化成文字，與推文拼接後送給 tokenizer，模擬真實業務場景。

In [ ]:
SELECT_COLS = [
    "airline",
    "airline_sentiment_confidence",
    "negativereason",
    "negativereason_confidence",
    "text",
    "retweet_count",
    "label",
]

ds = ds.select_columns(SELECT_COLS)

# 補空缺值：negativereason 填 'Unknown'，信心度填欄位平均值
# 注意：平均值在此處計算沒有洩漏問題，因為這是描述性統計，不是模型參數
ds = ds.map(
    lambda ex: {
        "negativereason": ex["negativereason"] if ex["negativereason"] is not None else "Unknown",
        "negativereason_confidence": (
            ex["negativereason_confidence"]
            if ex["negativereason_confidence"] is not None
            else 0.0  # placeholder；稍後用訓練集均值覆蓋
        ),
    }
)

print("處理後欄位：", ds.column_names)
print("筆數：", len(ds))

### 3-3  Train / Test 分割（含 Stratify）

**為什麼要 stratify？**  
本資料集嚴重不平衡（negative ~62%、neutral ~21%、positive ~17%）。若隨機切割，測試集可能恰好缺少 positive 樣本，導致評測結果不具代表性。`stratify_by_column` 保證每個 split 的類別比例相同。

**修正資料洩漏（Data Leakage）的正確做法：**  
先切分 → 用訓練集 `fit` scaler → 訓練集 `transform`、測試集只 `transform`。  
若在切分前對整個資料集執行 `fit_transform`，等於把測試集的最大／最小值資訊洩漏給了模型，會高估泛化能力。

In [ ]:
# 先切分，再做特徵工程
split = ds.train_test_split(test_size=0.1, stratify_by_column="label", seed=42)
train_ds = split["train"]
test_ds  = split["test"]

print("Train size:", len(train_ds))
print("Test  size:", len(test_ds))

# 確認分層結果
train_labels = train_ds["label"]
test_labels  = test_ds["label"]
for k, v in LABEL_MAP.items():
    tr = train_labels.count(v) / len(train_labels)
    te = test_labels.count(v)  / len(test_labels)
    print(f"  {k:8s}  train={tr:.2%}  test={te:.2%}")

In [ ]:
# MinMaxScaler 只對訓練集 fit，測試集只 transform（修正資料洩漏）
NUM_COLS = ["airline_sentiment_confidence", "negativereason_confidence", "retweet_count"]

scaler = MinMaxScaler()

# 取出 numpy array
train_num = np.array([[ex[c] for c in NUM_COLS] for ex in train_ds])
test_num  = np.array([[ex[c] for c in NUM_COLS] for ex in test_ds])

# 只用訓練集 fit
scaler.fit(train_num)
train_num_scaled = scaler.transform(train_num)
test_num_scaled  = scaler.transform(test_num)

# 把縮放後的值寫回 dataset
def apply_scaled(dataset, scaled_arr):
    """Inject scaled numerical columns back into a HuggingFace Dataset."""
    scaled_dict = {c: scaled_arr[:, i].tolist() for i, c in enumerate(NUM_COLS)}
    for col, vals in scaled_dict.items():
        dataset = dataset.add_column(col + "_scaled", vals)
    return dataset

train_ds = apply_scaled(train_ds, train_num_scaled)
test_ds  = apply_scaled(test_ds,  test_num_scaled)

print("新欄位：", [c for c in train_ds.column_names if c.endswith("_scaled")])

## Step 4  Tokenizer 與 Dataset 管線

### Padding 策略：動態 padding

```python
# map 階段不 padding，collator 在每個 batch 內對齊到該 batch 的最長序列
tokenizer(text, max_length=256, truncation=True)  # 不加 padding
collator = DataCollatorWithPadding(tokenizer)      # collate 時動態補齊
```

動態 padding 在序列長度差異大時（本資料集推文長度 10-200 token 不等）可減少 30-50% 計算量。

### 為什麼選 `hfl/rbt3`？

`rbt3` 是 3 層輕量 RoBERTa-wwm，適合中文文本；本資料集雖為英文推文，仍沿用原 notebook 的模型選擇，示範多特徵文字拼接的技巧。若換英文任務可改用 `distilbert-base-uncased`。

In [ ]:
MODEL_ID = "hfl/rbt3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def process_function(examples):
    """Combine multi-field features into a single string and tokenize."""
    combined_text = [
        (
            str(airline)
            + " [SEP] " + str(reason)
            + " [SEP] " + text
            + " sentiment_confidence: " + f"{sent_conf:.4f}"
            + " reason_confidence: "   + f"{reason_conf:.4f}"
            + " retweet_count: "       + f"{retweet:.4f}"
        )
        for airline, reason, text, sent_conf, reason_conf, retweet in zip(
            examples["airline"],
            examples["negativereason"],
            examples["text"],
            examples["airline_sentiment_confidence_scaled"],
            examples["negativereason_confidence_scaled"],
            examples["retweet_count_scaled"],
        )
    ]
    tokenized = tokenizer(combined_text, max_length=256, truncation=True)
    tokenized["labels"] = examples["label"]
    return tokenized

# batched=True 讓 tokenizer 批次處理，速度比逐筆快 3-5 倍
REMOVE_COLS = train_ds.column_names  # 移除所有原始欄位，只留 input_ids / attention_mask / labels

tokenized_train = train_ds.map(
    process_function,
    batched=True,
    num_proc=1,
    remove_columns=REMOVE_COLS,
    desc="Tokenizing train",
)
tokenized_test = test_ds.map(
    process_function,
    batched=True,
    num_proc=1,
    remove_columns=REMOVE_COLS,
    desc="Tokenizing test",
)

print("Tokenized train features:", tokenized_train.features)
print("Train sample:", tokenized_train[0].keys())

In [ ]:
# DataCollatorWithPadding：在 collate 時才對 batch 內做動態 padding
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

## Step 5  載入模型

2026 統一裝置載入慣例：

```python
cls_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config,
    device_map="auto",           # 自動分配：GPU → CPU → disk offload
    torch_dtype=torch.bfloat16, # 節省約一半 VRAM
    use_safetensors=True,        # 安全格式，載入速度比 pickle 快
)
```

**為什麼選 `bfloat16` 而非 `float16`？**  
- `bfloat16` 數值範圍與 `float32` 相同（指數位 8 bit），訓練穩定，不容易出現梯度下溢  
- `float16` 精度略高但動態範圍窄，需搭配 `GradScaler`，設定更複雜  
- A100 / H100 對 bf16 有原生硬體支援（Ampere+ 架構），速度等同 fp16

**`device_map='auto'` 的語意：**  
- 有多張 GPU → 自動切層分佈  
- 只有一張 GPU → 全部放 GPU  
- 記憶體不足 → 自動 offload 部分層到 CPU 或 disk（需 accelerate）  

**`use_safetensors=True` 的好處：**  
- 相對 pickle（`.bin`），safetensors 不執行任意 Python 程式碼，載入更安全  
- 直接 mmap 映射檔案，大模型載入速度快 2-3 倍

> **VRAM 提示（rbt3）：** rbt3 約 38M 參數，bf16 佔用約 76 MB，一般 GPU 均可執行。
> 若改用大型模型（如 BERT-large），bf16 約需 700 MB；可考慮加 BitsAndBytesConfig 4-bit 量化。

In [ ]:
config = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=len(LABEL_MAP),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# 2026 統一載入慣例：device_map + torch_dtype + use_safetensors
# rbt3 很小，use_safetensors 若找不到 .safetensors 會自動 fallback 到 .bin
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True,  # classifier head 層數改變時必須加
)

print("Model config id2label:", model.config.id2label)
print("Classifier shape:", model.classifier.weight.shape)

## Step 6  評測函式

`evaluate.combine` 將多個指標合併為單一物件，由 `Trainer` 在每個 `eval_strategy` 步驟自動呼叫 `compute_metrics`，不需手寫迴圈。

```python
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return clf_metrics.compute(predictions=preds, references=labels, average="macro")
```

In [ ]:
# 合併四個指標為一個 metric 物件
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    """
    Callback used by Trainer.evaluate() and Trainer.train().
    eval_pred is a named tuple: (logits, labels) as numpy arrays.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    result = clf_metrics.compute(
        predictions=preds,
        references=labels,
        average="macro",  # macro 對各類別一視同仁，不受類別數量影響
    )
    return result

## Step 7  TrainingArguments

### 完整 2026 必填項目說明

| 參數 | 值 | 理由 |
|---|---|---|
| `bf16=True` | True | 節省 VRAM，訓練穩定（見 Step 5） |
| `optim` | `adamw_torch_fused` | 融合 CUDA kernel，比原版 AdamW 快 10-20%；比 Adam 有正確 weight decay |
| `warmup_ratio` | 0.1 | 前 10% 步驟線性 warmup，避免學習率過早衝太高造成發散 |
| `lr_scheduler_type` | `cosine` | Cosine 退火讓後期學習率平滑下降，通常優於 linear |
| `max_grad_norm` | 1.0 | Gradient clipping 防止梯度爆炸 |
| `eval_strategy` | `steps` | 訓練中定期評測，搭配 `load_best_model_at_end` 選最佳 checkpoint |
| `save_safetensors` | True | 儲存 safetensors 格式（2026 標準） |
| `seed` | 42 | 結合 `set_seed(42)` 確保結果可重現 |

**Effective batch size = `per_device_train_batch_size` × `gradient_accumulation_steps` × GPU 數**  
本例：32 × 2 × 1 = 64。

In [ ]:
OUTPUT_DIR = "./tweet-sentiment-rbt3"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # --- 訓練規模 ---
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,   # effective batch = 32 * 2 = 64
    # --- 優化器與學習率 ---
    learning_rate=2e-5,
    optim="adamw_torch_fused",       # AdamW（含 weight decay），fused CUDA kernel
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    # --- 精度 ---
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    # --- 評測與儲存 ---
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    save_safetensors=True,
    # --- 可重現性 ---
    seed=42,
    data_seed=42,
    # --- 日誌 ---
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    report_to="none",  # 關閉 wandb / tensorboard（按需開啟）
)

print("TrainingArguments 設定完成")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

## Step 8  訓練

`Trainer` 封裝裝置搬移、混合精度、梯度累積、checkpoint 儲存與最佳模型選取，`TrainingArguments` 統一管理所有超參數。

```python
trainer = Trainer(
    model=model,
    args=training_args,      # bf16、AdamW、cosine scheduler 等全包
    train_dataset=...,
    eval_dataset=...,
    data_collator=collator,  # 動態 padding
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()
```

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),  # 驗證 f1 連續 3 次不進步就停
    ],
)

train_result = trainer.train()
print("Training finished.")
print(train_result.metrics)

## Step 9  評測結果

In [ ]:
# 對測試集跑完整評測
metrics = trainer.evaluate(tokenized_test)
print("\n=== Test Set Metrics ===")
for k, v in metrics.items():
    print(f"  {k:<30s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

### 混淆矩陣（錯誤分析）

混淆矩陣讓我們看出模型在哪些類別之間容易混淆，比單一 F1 數字更有診斷價值。

In [ ]:
import numpy as np

# 取得預測結果
pred_output = trainer.predict(tokenized_test)
logits      = pred_output.predictions
labels      = pred_output.label_ids
preds       = np.argmax(logits, axis=-1)

# 建立混淆矩陣
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(labels, preds)
print("Confusion Matrix (rows=true, cols=predicted):")
print("Labels:", list(ID2LABEL.values()))
print(cm)
print()
print(classification_report(labels, preds, target_names=list(ID2LABEL.values())))

## Step 10  單筆推論

推論時同樣使用 `DataCollatorWithPadding`，保持訓練/推論一致。

In [ ]:
def build_combined_text(airline, reason, text, sent_conf, reason_conf, retweet):
    """Reproduce the same feature-concatenation format used during training."""
    return (
        str(airline)
        + " [SEP] " + str(reason)
        + " [SEP] " + str(text)
        + " sentiment_confidence: "  + f"{float(sent_conf):.4f}"
        + " reason_confidence: "     + f"{float(reason_conf):.4f}"
        + " retweet_count: "         + f"{float(retweet):.4f}"
    )


def predict_single(example_idx: int):
    """Run inference on a single test example by index."""
    raw = test_ds[example_idx]
    text = build_combined_text(
        raw["airline"],
        raw["negativereason"],
        raw["text"],
        raw["airline_sentiment_confidence_scaled"],
        raw["negativereason_confidence_scaled"],
        raw["retweet_count_scaled"],
    )

    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.inference_mode():
        logits = model(**inputs).logits
        pred_id = torch.argmax(logits, dim=-1).item()

    print(f"Input text    : {raw['text']}")
    print(f"True label    : {ID2LABEL[raw['label']]}")
    print(f"Predicted     : {ID2LABEL[pred_id]}")
    print(f"Confidence    : {torch.softmax(logits, dim=-1).squeeze().tolist()}")


# 測試幾筆
for idx in [0, 50, 200]:
    print(f"\n--- Test example #{idx} ---")
    predict_single(idx)

## Step 11  儲存模型與 Model Card

### 為什麼要儲存 `id2label` / `label2id`？

若只儲存 weights，重新載入時 `config.num_labels=3` 但不知道 0/1/2 對應什麼情緒，Pipeline 的 `label` 輸出會是 `LABEL_0`。將 `id2label` 寫入 `config.json`，任何人載入都能立即看到語意標籤。

In [ ]:
# 儲存最佳 checkpoint（含 id2label / label2id）
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")
print("id2label:", model.config.id2label)

In [ ]:
# （可選）推送到 HuggingFace Hub
# 需先執行 `huggingface-cli login` 或設定 HF_TOKEN 環境變數
#
# HUB_MODEL_ID = "your-username/tweet-sentiment-rbt3"
# trainer.push_to_hub(
#     model_id=HUB_MODEL_ID,
#     tags=["text-classification", "sentiment-analysis", "twitter", "zh"],
#     language=["zh", "en"],
#     license="apache-2.0",
#     finetuned_from=MODEL_ID,
#     tasks=["text-classification"],
# )

## Step 12  從儲存的目錄重新載入（驗證可重現性）

In [ ]:
# 重新載入並做一次推論，確認 id2label 正確持久化
loaded_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
loaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

print("Loaded id2label:", loaded_model.config.id2label)

# 快速測試
sample_text = "@UnitedAirlines My flight was cancelled and no one helped me. Very disappointed."
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=256)
with torch.inference_mode():
    logits = loaded_model(**inputs).logits
pred = torch.argmax(logits, dim=-1).item()
print(f"\nSample: {sample_text}")
print(f"Prediction: {loaded_model.config.id2label[pred]}")

## 小結

### 本 Notebook 做了什麼

1. **資料來源**：從 HuggingFace Hub (`osanseviero/twitter-airline-sentiment`) 載入，取代 Google Drive 依賴
2. **修正資料洩漏**：`MinMaxScaler.fit()` 只對訓練集執行，測試集只做 `transform`
3. **Stratified split**：`train_test_split(stratify_by_column='label', seed=42)` 保留類別比例
4. **動態 padding**：`DataCollatorWithPadding` 在 batch 內動態對齊，減少計算浪費
5. **2026 載入慣例**：`device_map='auto'` + `torch_dtype=torch.bfloat16` + `use_safetensors=True`
6. **Trainer 取代手刻迴圈**：`AdamW fused`、cosine scheduler、warmup、gradient clipping 全部由 `TrainingArguments` 管理
7. **評測整合**：`evaluate.combine` + `compute_metrics` + 混淆矩陣分析
8. **Model card 持久化**：`id2label` / `label2id` 隨 config 儲存

### 練習題

1. 把 `MODEL_ID` 換成 `distilbert-base-uncased`，重新訓練並比較 F1 分數，觀察英文 pretrained 模型在這個英文資料集上的表現差異
2. 在 `process_function` 中移除數值特徵（只保留 `text`），觀察 F1 變化——這幫助你判斷附加特徵的實際貢獻
3. 修改 `num_train_epochs=1` + `gradient_accumulation_steps=4`，確認 effective batch size 不變的情況下訓練結果是否接近
4. 將混淆矩陣視覺化（`seaborn.heatmap`），找出模型最容易把「negative」誤判為哪個類別，並思考特徵工程的改進方向
5. 取消 `push_to_hub` 的註解，將模型推送到自己的 Hub 帳戶，並用 `pipeline('text-classification', model='your-username/tweet-sentiment-rbt3')` 驗證端到端流程